# Deep Learning for Skin Lesion Triage

**AUTHOR**  
Rodrigo Kang

## Overview

This notebook develops a PyTorch-based binary computer vision study using the HAM10000 / ISIC 2018 Task 3 training data. The analytical objective is to investigate whether dermoscopic images can support **experimental prioritisation for specialist dermatological review**.

The source dataset contains seven diagnostic categories. This project derives a separate binary target in which `mel`, `bcc`, and `akiec` are grouped as lesions designated for priority review, while `nv`, `bkl`, `df`, and `vasc` form the lower-priority group. This grouping is a project-specific modelling decision: it is not an original dataset label, a clinically validated referral rule, an autonomous diagnostic system, or a substitute for dermatologist assessment.

HAM10000 combines images acquired from more than one source and includes different ground-truth procedures, including histopathology, follow-up, expert consensus, and confocal microscopy. Some lesions have multiple images, so later splitting must operate at lesion level rather than image level. The dataset is also markedly imbalanced and heterogeneous in acquisition conditions. These properties make benchmark performance informative for controlled experimentation, but insufficient evidence of clinical validity or external generalisation.

## Part 1 — Setup and Reproducibility

Part 1 establishes the notebook configuration, binary target definition, relative paths, output folders, reproducibility controls, environment reporting, and basic source-path checks. It does not extract the image archive, load the full metadata table, audit the data, construct splits, or train a model.

### Imports

Only libraries required for the initial setup are imported here. Additional approved dependencies will be introduced in later sections when they are first needed.

In [1]:
from __future__ import annotations

import os
import platform
import random
import sys
from pathlib import Path
from typing import Mapping

import numpy as np
import torch
import torchvision

### Central Configuration

Configuration values are defined in one place so that later stages use the same seed, paths, target mapping, and output locations. All executable paths are relative to the `python/` notebook directory.

In [2]:
project_name = "deep-learning-skin-lesion-triage"
random_seed = 42

image_archive_path = Path("../data/ISIC-images.zip")
metadata_path = Path("../data/challenge-2018-task-3-training_metadata_2026-07-21.csv")

output_root = Path("output")
output_directories = {
    "figures": output_root / "figures",
    "tables": output_root / "tables",
    "models": output_root / "models",
    "metadata": output_root / "metadata",
}

source_paths = {
    "image_archive": image_archive_path,
    "metadata": metadata_path,
}

### Binary Target Definition

The seven source diagnosis labels remain conceptually distinct from the derived binary target. A positive model output will represent the project-defined `priority dermatological review` group; it must not be interpreted as a diagnosis or as proof of clinical urgency for an individual lesion.

In [3]:
priority_review_labels = {"mel", "bcc", "akiec"}
lower_priority_labels = {"nv", "bkl", "df", "vasc"}

binary_class_names = {
    0: "lower-priority lesion",
    1: "priority dermatological review",
}

diagnosis_to_binary_target = {
    **{label: 1 for label in priority_review_labels},
    **{label: 0 for label in lower_priority_labels},
}

expected_source_labels = priority_review_labels | lower_priority_labels

assert priority_review_labels.isdisjoint(lower_priority_labels)
assert len(expected_source_labels) == 7
assert set(diagnosis_to_binary_target) == expected_source_labels
assert set(diagnosis_to_binary_target.values()) == {0, 1}

### Setup Helpers

The following helpers are limited to tasks required at notebook start-up: creating output folders, applying reproducibility settings, and checking that configured source files are present.

In [4]:
def create_output_directories(directories: Mapping[str, Path]) -> None:
    """
    Create the configured local output directories when they do not exist.

    Input:
    ------
    directories : Mapping[str, Path]
        Named output directory paths to create.

    Outputs:
    --------
    None
        The directories are created on the local file system.

    Author:
    -------
    Rodrigo Kang
    """
    for directory in directories.values():
        directory.mkdir(parents=True, exist_ok=True)


def configure_reproducibility(seed: int) -> None:
    """
    Configure fixed random seeds and deterministic PyTorch behaviour where practical.

    Input:
    ------
    seed : int
        Seed applied to Python, NumPy, and PyTorch random number generators.

    Outputs:
    --------
    None
        Reproducibility settings are applied to the active Python process.

    Author:
    -------
    Rodrigo Kang
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def validate_source_paths(paths: Mapping[str, Path]) -> dict[str, bool]:
    """
    Check that each configured source path exists and points to a file.

    Input:
    ------
    paths : Mapping[str, Path]
        Named source file paths to validate.

    Outputs:
    --------
    validation_results : dict[str, bool]
        Boolean file-status result for each configured source path.

    Author:
    -------
    Rodrigo Kang
    """
    validation_results = {
        name: path.exists() and path.is_file()
        for name, path in paths.items()
    }

    missing_paths = [
        str(paths[name])
        for name, is_valid in validation_results.items()
        if not is_valid
    ]
    if missing_paths:
        missing_list = "\n- ".join(missing_paths)
        raise FileNotFoundError(
            "Required source files were not found at the configured relative paths:"
            f"\n- {missing_list}"
        )

    return validation_results

### Initialisation and Environment

The initialisation cell creates the local output structure, applies reproducibility controls, selects the available compute device, reports key versions, and performs file-presence checks only. Exact bitwise reproducibility can still vary across hardware, operating systems, drivers, and library versions.

In [5]:
create_output_directories(output_directories)
configure_reproducibility(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
source_path_status = validate_source_paths(source_paths)

environment_summary = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "selected_device": str(device),
}

print("Environment")
for key, value in environment_summary.items():
    print(f"- {key}: {value}")

print("\nConfigured source files")
for name, path in source_paths.items():
    print(f"- {name}: {path} ({'found' if source_path_status[name] else 'missing'})")

print("\nOutput directories")
for name, path in output_directories.items():
    print(f"- {name}: {path}")

Environment
- python: 3.12.13
- platform: Windows-10-10.0.19045-SP0
- numpy: 2.5.1
- torch: 2.13.0+cpu
- torchvision: 0.28.0+cpu
- cuda_available: False
- selected_device: cpu

Configured source files
- image_archive: ..\data\ISIC-images.zip (found)
- metadata: ..\data\challenge-2018-task-3-training_metadata_2026-07-21.csv (found)

Output directories
- figures: output\figures
- tables: output\tables
- models: output\models
- metadata: output\metadata


## Part 2 — Metadata and Image Archive Audit


Part 2 examines the supplied metadata and compressed image archive without extracting the images. It identifies the source diagnosis fields, constructs the project-specific binary target, checks metadata and archive consistency, and records the main integrity findings required before any split or modelling work. No train, validation or test partitions are created here.


### Data Sources and Provenance


The HAM10000 training set contains dermoscopic images assembled from more than one source and over an extended collection period. Ground truth was established through several routes, including histopathology, longitudinal stability, expert consensus and confocal microscopy. Some lesions have more than one image, including different views or magnifications, so later partitioning must operate at lesion level rather than image level.

The ISIC 2018 disease-classification challenge used the seven HAM10000 categories as a benchmark task. Its internal and external test partitions also showed that similar aggregate benchmark results can conceal different generalisation behaviour across acquisition sources. These properties make the dataset suitable for a controlled modelling study, but they do not establish clinical validity or transportability to new services, devices or patient populations.


### Audit Imports


The audit adds only the approved libraries and standard-library modules needed to inspect tabular metadata, read image headers directly from the ZIP archive, and save compact audit artefacts.


In [6]:
import json
import zipfile
from collections import Counter
from io import BytesIO

import pandas as pd
from PIL import Image, UnidentifiedImageError


### Metadata Loading and Schema Checks


The metadata is loaded once and retained in `metadata`. Required identifiers and diagnostic hierarchy fields are checked explicitly. The diagnosis mapping is derived from the supplied hierarchy rather than assuming that the short HAM10000 labels are already present.


In [7]:
required_metadata_columns = {
    "isic_id",
    "lesion_id",
    "diagnosis_2",
    "diagnosis_3",
    "diagnosis_confirm_type",
    "age_approx",
    "sex",
    "anatom_site_1",
}


def load_metadata(csv_path: Path, required_columns: set[str]) -> pd.DataFrame:
    """
    Load metadata and verify that required columns are available.

    Input:
    ------
    csv_path : Path
        Relative path to the source metadata CSV file.
    required_columns : set[str]
        Columns required for the initial audit and target construction.

    Outputs:
    --------
    metadata_frame : pd.DataFrame
        Loaded metadata with leading and trailing column-name whitespace removed.

    Author:
    -------
    Rodrigo Kang
    """
    metadata_frame = pd.read_csv(csv_path)
    metadata_frame.columns = metadata_frame.columns.str.strip()

    missing_columns = sorted(required_columns - set(metadata_frame.columns))
    if missing_columns:
        raise ValueError(f"Required metadata columns are missing: {missing_columns}")

    return metadata_frame


metadata = load_metadata(metadata_path, required_metadata_columns)

print(f"Metadata rows: {len(metadata):,}")
print(f"Metadata columns: {metadata.shape[1]}")
print("Columns:")
print(metadata.columns.tolist())


Metadata rows: 10,015
Metadata columns: 17
Columns:
['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_3', 'anatom_site_special', 'concomitant_biopsy', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 'lesion_id', 'melanocytic', 'sex']


### Source Diagnosis and Binary Target Construction


The short source labels are reconstructed transparently from the diagnostic hierarchy. `akiec` combines records labelled as squamous cell carcinoma or solar/actinic keratosis in the supplied hierarchy, matching the corresponding HAM10000 challenge category. Vascular lesions are identified from `diagnosis_2` because `diagnosis_3` is not populated for those rows. The original hierarchy remains unchanged for later subgroup analysis.


In [8]:
diagnosis_3_to_source_label = {
    "Nevus": "nv",
    "Melanoma, NOS": "mel",
    "Pigmented benign keratosis": "bkl",
    "Basal cell carcinoma": "bcc",
    "Squamous cell carcinoma, NOS": "akiec",
    "Solar or actinic keratosis": "akiec",
    "Dermatofibroma": "df",
}
vascular_diagnosis_2 = "Benign soft tissue proliferations - Vascular"


def derive_source_diagnosis(metadata_frame: pd.DataFrame) -> pd.Series:
    """
    Reconstruct the seven HAM10000 source labels from the diagnosis hierarchy.

    Input:
    ------
    metadata_frame : pd.DataFrame
        Metadata containing `diagnosis_2` and `diagnosis_3` fields.

    Outputs:
    --------
    source_diagnosis : pd.Series
        Seven-class source diagnosis label for each metadata row.

    Author:
    -------
    Rodrigo Kang
    """
    source_diagnosis = metadata_frame["diagnosis_3"].map(diagnosis_3_to_source_label)
    vascular_mask = metadata_frame["diagnosis_2"].eq(vascular_diagnosis_2)
    source_diagnosis = source_diagnosis.mask(vascular_mask, "vasc")
    return source_diagnosis


metadata = metadata.copy()
metadata["source_diagnosis"] = derive_source_diagnosis(metadata)
metadata["binary_target"] = metadata["source_diagnosis"].map(diagnosis_to_binary_target)

observed_source_labels = set(metadata["source_diagnosis"].dropna().unique())
unexpected_source_labels = observed_source_labels - expected_source_labels
missing_expected_labels = expected_source_labels - observed_source_labels

assert not unexpected_source_labels, unexpected_source_labels
assert not missing_expected_labels, missing_expected_labels
assert metadata["source_diagnosis"].notna().all()
assert metadata["binary_target"].notna().all()
assert set(metadata["binary_target"].unique()) == {0, 1}

metadata["binary_target"] = metadata["binary_target"].astype("int64")

source_diagnosis_counts = (
    metadata["source_diagnosis"]
    .value_counts()
    .rename_axis("source_diagnosis")
    .reset_index(name="image_count")
)
binary_target_counts = (
    metadata["binary_target"]
    .value_counts()
    .sort_index()
    .rename_axis("binary_target")
    .reset_index(name="image_count")
)
binary_target_counts["class_name"] = binary_target_counts["binary_target"].map(binary_class_names)
binary_target_counts["proportion"] = binary_target_counts["image_count"] / len(metadata)

print("Source diagnosis counts")
print(source_diagnosis_counts.to_string(index=False))
print()
print("Derived binary target counts")
print(binary_target_counts.to_string(index=False))


Source diagnosis counts
source_diagnosis  image_count
              nv         6705
             mel         1113
             bkl         1099
             bcc          514
           akiec          327
            vasc          142
              df          115

Derived binary target counts
 binary_target  image_count                     class_name  proportion
             0         8061          lower-priority lesion    0.804893
             1         1954 priority dermatological review    0.195107


### Metadata Integrity Audit


The metadata audit checks row-level duplication, uniqueness of image identifiers, lesion multiplicity, missingness and available provenance fields. Counts are descriptive at this stage; no records are removed and no imputation is performed.


In [9]:
duplicate_row_count = int(metadata.duplicated().sum())
duplicate_image_id_count = int(metadata["isic_id"].duplicated().sum())
missing_lesion_id_count = int(metadata["lesion_id"].isna().sum())

images_per_lesion = metadata.groupby("lesion_id", dropna=False).size()
lesion_count = int(images_per_lesion.size)
lesions_with_multiple_images = int((images_per_lesion > 1).sum())
images_from_repeated_lesions = int(images_per_lesion[images_per_lesion > 1].sum())
maximum_images_per_lesion = int(images_per_lesion.max())

missing_value_summary = (
    metadata.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_proportion=lambda frame: frame["missing_count"] / len(metadata))
    .query("missing_count > 0")
    .sort_values(["missing_count", "missing_proportion"], ascending=False)
    .reset_index(names="column")
)

confirmation_counts = (
    metadata["diagnosis_confirm_type"]
    .value_counts(dropna=False)
    .rename_axis("diagnosis_confirm_type")
    .reset_index(name="image_count")
)

metadata_integrity_summary = pd.DataFrame(
    {
        "measure": [
            "metadata_rows",
            "metadata_columns",
            "unique_image_ids",
            "duplicate_rows",
            "duplicate_image_ids",
            "unique_lesions",
            "missing_lesion_ids",
            "lesions_with_multiple_images",
            "images_from_repeated_lesions",
            "maximum_images_per_lesion",
        ],
        "value": [
            len(metadata),
            metadata.shape[1],
            metadata["isic_id"].nunique(),
            duplicate_row_count,
            duplicate_image_id_count,
            lesion_count,
            missing_lesion_id_count,
            lesions_with_multiple_images,
            images_from_repeated_lesions,
            maximum_images_per_lesion,
        ],
    }
)

print(metadata_integrity_summary.to_string(index=False))
print()
print("Ground-truth confirmation counts")
print(confirmation_counts.to_string(index=False))
print()
print("Columns with missing values")
print(missing_value_summary.to_string(index=False))


                     measure  value
               metadata_rows  10015
            metadata_columns     19
            unique_image_ids  10015
              duplicate_rows      0
         duplicate_image_ids      0
              unique_lesions   7470
          missing_lesion_ids      0
lesions_with_multiple_images   1956
images_from_repeated_lesions   4501
   maximum_images_per_lesion      6

Ground-truth confirmation counts
                       diagnosis_confirm_type  image_count
                               histopathology         5340
             serial imaging showing no change         3704
                single image expert consensus          902
confocal microscopy with consensus dermoscopy           69

Columns with missing values
             column  missing_count  missing_proportion
anatom_site_special           9551            0.953669
      anatom_site_3           7657            0.764553
      anatom_site_2           4840            0.483275
      anatom_site_1       

### Compressed Image Archive Audit


Images are inspected directly inside the ZIP archive, avoiding extraction and duplicate local storage. The audit reconciles image identifiers with the metadata and reads each image sufficiently to record format, dimensions and colour mode while identifying unreadable members. Non-image licence and attribution files are retained but excluded from image counts.


In [10]:
supported_image_extensions = {".jpg", ".jpeg", ".png"}


def audit_image_archive(archive_path: Path) -> tuple[pd.DataFrame, list[str]]:
    """
    Inspect image members in a ZIP archive without extracting them.

    Input:
    ------
    archive_path : Path
        Relative path to the compressed image archive.

    Outputs:
    --------
    image_audit : pd.DataFrame
        One row per image member with identifier, format, dimensions, mode and read status.
    non_image_members : list[str]
        Archive members excluded from the image audit because they are not image files.

    Author:
    -------
    Rodrigo Kang
    """
    audit_records = []
    non_image_members = []

    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue

            member_path = Path(member.filename)
            if member_path.suffix.lower() not in supported_image_extensions:
                non_image_members.append(member.filename)
                continue

            record = {
                "archive_member": member.filename,
                "isic_id": member_path.stem,
                "extension": member_path.suffix.lower(),
                "file_size_bytes": member.file_size,
                "image_format": None,
                "width": None,
                "height": None,
                "mode": None,
                "readable": False,
                "error": None,
            }

            try:
                with archive.open(member) as image_file:
                    image_bytes = image_file.read()
                with Image.open(BytesIO(image_bytes)) as image:
                    image.load()
                    record.update(
                        {
                            "image_format": image.format,
                            "width": image.width,
                            "height": image.height,
                            "mode": image.mode,
                            "readable": True,
                        }
                    )
            except (OSError, ValueError, UnidentifiedImageError) as error:
                record["error"] = str(error)

            audit_records.append(record)

    return pd.DataFrame(audit_records), sorted(non_image_members)


image_audit, non_image_archive_members = audit_image_archive(image_archive_path)

metadata_image_ids = set(metadata["isic_id"])
archive_image_ids = set(image_audit["isic_id"])
metadata_rows_without_images = sorted(metadata_image_ids - archive_image_ids)
archive_images_without_metadata = sorted(archive_image_ids - metadata_image_ids)

image_dimension_counts = (
    image_audit.groupby(["width", "height"], dropna=False)
    .size()
    .reset_index(name="image_count")
    .sort_values("image_count", ascending=False)
)
image_mode_counts = (
    image_audit["mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .reset_index(name="image_count")
)
image_format_counts = (
    image_audit["image_format"]
    .value_counts(dropna=False)
    .rename_axis("image_format")
    .reset_index(name="image_count")
)

archive_integrity_summary = pd.DataFrame(
    {
        "measure": [
            "archive_image_members",
            "unique_archive_image_ids",
            "unreadable_images",
            "metadata_rows_without_images",
            "archive_images_without_metadata",
            "non_image_archive_members",
        ],
        "value": [
            len(image_audit),
            image_audit["isic_id"].nunique(),
            int((~image_audit["readable"]).sum()),
            len(metadata_rows_without_images),
            len(archive_images_without_metadata),
            len(non_image_archive_members),
        ],
    }
)

print(archive_integrity_summary.to_string(index=False))
print()
print("Image formats")
print(image_format_counts.to_string(index=False))
print()
print("Image modes")
print(image_mode_counts.to_string(index=False))
print()
print("Image dimensions")
print(image_dimension_counts.to_string(index=False))
print()
print("Non-image archive members")
for member_name in non_image_archive_members:
    print(f"- {member_name}")


                        measure  value
          archive_image_members  10015
       unique_archive_image_ids  10015
              unreadable_images      0
   metadata_rows_without_images      0
archive_images_without_metadata      0
      non_image_archive_members      3

Image formats
image_format  image_count
        JPEG        10015

Image modes
mode  image_count
 RGB        10015

Image dimensions
 width  height  image_count
   600     450        10015

Non-image archive members
- attribution.txt
- licenses/CC-BY-NC.txt
- metadata.csv


### Audit Artefacts and Findings


Compact CSV and JSON artefacts are saved for traceability. The derived target is stored alongside the original diagnostic hierarchy, while the full per-image archive audit records file integrity and image properties. These files support later split construction without modifying the source data.


In [11]:
target_definition = {
    "project_name": project_name,
    "target_name": "binary_target",
    "positive_class_index": 1,
    "positive_class_name": binary_class_names[1],
    "positive_source_labels": sorted(priority_review_labels),
    "negative_class_index": 0,
    "negative_class_name": binary_class_names[0],
    "negative_source_labels": sorted(lower_priority_labels),
    "source_label_column": "source_diagnosis",
    "statement": (
        "Project-specific analytical grouping for experimental prioritisation; "
        "not an original dataset ground truth or clinically validated referral rule."
    ),
}

metadata_audit_path = output_directories["tables"] / "metadata-with-derived-target.csv"
source_counts_path = output_directories["tables"] / "source-diagnosis-counts.csv"
binary_counts_path = output_directories["tables"] / "binary-target-counts.csv"
missing_values_path = output_directories["tables"] / "missing-value-summary.csv"
confirmation_counts_path = output_directories["tables"] / "ground-truth-confirmation-counts.csv"
image_audit_path = output_directories["tables"] / "image-archive-audit.csv"
audit_summary_path = output_directories["metadata"] / "data-audit-summary.json"
target_definition_path = output_directories["metadata"] / "binary-target-definition.json"

metadata.to_csv(metadata_audit_path, index=False)
source_diagnosis_counts.to_csv(source_counts_path, index=False)
binary_target_counts.to_csv(binary_counts_path, index=False)
missing_value_summary.to_csv(missing_values_path, index=False)
confirmation_counts.to_csv(confirmation_counts_path, index=False)
image_audit.to_csv(image_audit_path, index=False)

audit_summary = {
    "metadata": dict(zip(metadata_integrity_summary["measure"], metadata_integrity_summary["value"].astype(int))),
    "archive": dict(zip(archive_integrity_summary["measure"], archive_integrity_summary["value"].astype(int))),
    "source_diagnosis_counts": dict(zip(source_diagnosis_counts["source_diagnosis"], source_diagnosis_counts["image_count"].astype(int))),
    "binary_target_counts": dict(zip(binary_target_counts["binary_target"].astype(str), binary_target_counts["image_count"].astype(int))),
    "metadata_rows_without_images": metadata_rows_without_images,
    "archive_images_without_metadata": archive_images_without_metadata,
    "non_image_archive_members": non_image_archive_members,
}

with audit_summary_path.open("w", encoding="utf-8") as file:
    json.dump(audit_summary, file, indent=2)
with target_definition_path.open("w", encoding="utf-8") as file:
    json.dump(target_definition, file, indent=2)

print("Saved audit artefacts")
for artefact_path in [
    metadata_audit_path,
    source_counts_path,
    binary_counts_path,
    missing_values_path,
    confirmation_counts_path,
    image_audit_path,
    audit_summary_path,
    target_definition_path,
]:
    print(f"- {artefact_path}")


Saved audit artefacts
- output\tables\metadata-with-derived-target.csv
- output\tables\source-diagnosis-counts.csv
- output\tables\binary-target-counts.csv
- output\tables\missing-value-summary.csv
- output\tables\ground-truth-confirmation-counts.csv
- output\tables\image-archive-audit.csv
- output\metadata\data-audit-summary.json
- output\metadata\binary-target-definition.json


The audit establishes whether the metadata and archive are internally aligned and quantifies the structural issues that matter for modelling. In particular, repeated images associated with the same lesion confirm that the next partitioning step must group by `lesion_id`. The derived target is intentionally analytical: it groups three source categories for prioritised review without implying common clinical urgency or replacing the original diagnosis labels.


## Exploratory Analysis (Optional)

This optional section presents exploratory analyses used to better understand the structure and characteristics of the data.

Typical elements may include:

- Descriptive statistics
- Visual exploration
- Correlation analysis
- Distribution analysis
- Preliminary pattern identification

## Methodology

This section implements the methodological framework discussed in the portfolio chapter.

Typical topics may include:

- Model definition
- Simulation procedures
- Optimization methods
- Statistical methodologies
- Experimental design

Short technical comments may be included when relevant to the implementation.

## Model Development

This section contains the practical implementation of the models and algorithms used throughout the project.

Typical topics may include:

- Model construction
- Training pipelines
- Hyperparameter configuration
- Numerical procedures
- Auxiliary algorithms

## Training and Validation

This section presents the procedures used to train, validate, and evaluate the models.

Typical topics may include:

- Training procedures
- Cross-validation strategies
- Performance monitoring
- Error estimation
- Model comparison

## Results and Evaluation

This section presents the main computational results obtained throughout the analysis.

Typical elements may include:

- Performance metrics
- Visualizations and plots
- Comparative analyses
- Error analysis
- Sensitivity analysis

The emphasis is placed on understanding model behavior and interpreting the practical implications of the results.

## Conclusions

This section summarizes the main findings and computational outcomes of the notebook.

The discussion may include:

- Main implementation results
- Practical observations
- Computational considerations
- Potential improvements